# AgroPredict Colombia
## Optimización y evaluación de modelos predictivos

**Despliegue de Soluciones Analíticas — Segunda entrega**

### Objetivo

Optimizar y comparar los modelos supervisados desarrollados
en `04_modelado.ipynb` para estimar el rendimiento municipal
del maíz en Colombia (t/ha).

### Metodología

1. Construir un modelo de referencia basado en el rendimiento histórico.
2. Implementar validación temporal con ventanas expansivas.
3. Optimizar hiperparámetros de Random Forest y XGBoost.
4. Comparar el desempeño mediante RMSE, MAE y R².
5. Registrar los experimentos en MLflow.

### Consideraciones metodológicas

- Entrenamiento y optimización: datos históricos hasta 2023.
- Validación: año 2024.
- Prueba final: año 2025, reservado hasta seleccionar el modelo.
- Se utilizarán únicamente variables disponibles antes de la siembra.
- Se conservará la separación temporal para evitar fuga de información.

In [1]:
# 1. Importación de librerías y configuración de rutas

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Identificar automáticamente la raíz del repositorio
cwd = Path.cwd().resolve()

try:
    ROOT = next(
        ruta for ruta in [cwd, *cwd.parents]
        if (ruta / ".git").exists() or (ruta / ".dvc").exists()
    )
except StopIteration:
    raise FileNotFoundError("No se encontró la raíz de AgroPredict Colombia.")

# Archivo generado por el notebook de ingeniería de características
RUTA_DATOS = ROOT / "data" / "processed" / "features_estacional.parquet"

print(f"Repositorio: {ROOT}")
print(f"Archivo de datos: {RUTA_DATOS}")
print(f"Archivo disponible: {RUTA_DATOS.exists()}")

Repositorio: /home/vickingo98/agropredict-colombia
Archivo de datos: /home/vickingo98/agropredict-colombia/data/processed/features_estacional.parquet
Archivo disponible: True


## 2. Carga y exploración de los datos para modelado

In [2]:
# Cargar el conjunto de características estacionales
df = pd.read_parquet(RUTA_DATOS)

print(f"Dimensiones del dataset: {df.shape[0]:,} filas y {df.shape[1]} columnas")

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nRegistros por año:")
print(df.groupby("anio").size())

print("\nValores faltantes por columna:")
print(df.isna().sum())

display(df.head())

Dimensiones del dataset: 14,935 filas y 12 columnas

Columnas disponibles:
['cod_departamento', 'departamento', 'cod_municipio', 'municipio', 'sistema_productivo', 'anio', 'semestre_b', 'area_cosechada', 'produccion', 'rendimiento', 't', 'rendimiento_rezago']

Registros por año:
anio
2019    2107
2020    2180
2021    2176
2022    2127
2023    2111
2024    2104
2025    2130
dtype: int64

Valores faltantes por columna:
cod_departamento         0
departamento             0
cod_municipio            0
municipio                0
sistema_productivo       0
anio                     0
semestre_b               0
area_cosechada           0
produccion               0
rendimiento              0
t                        0
rendimiento_rezago    2777
dtype: int64


,cod_departamento,departamento,cod_municipio,municipio,sistema_productivo,anio,semestre_b,area_cosechada,produccion,rendimiento,t,rendimiento_rezago
0,05,Antioquia,05002,Abejorral,tradicional,2019,0,100.0,200.0,2.0,4038,NaN
1,05,Antioquia,05002,Abejorral,tradicional,2019,1,75.0,150.0,2.0,4039,NaN
2,05,Antioquia,05002,Abejorral,tradicional,2020,0,100.0,200.0,2.0,4040,2.0
3,05,Antioquia,05002,Abejorral,tradicional,2020,1,75.0,150.0,2.0,4041,2.0
4,05,Antioquia,05002,Abejorral,tradicional,2021,0,98.0,196.0,2.0,4042,2.0


## 3. Preparación de los datos y validación temporal

In [3]:
# Conservar registros con rendimiento histórico disponible
df_modelo = df.dropna(subset=["rendimiento_rezago"]).copy()

# Verificar la distribución temporal
resumen_anual = pd.DataFrame({
    "Registros originales": df.groupby("anio").size(),
    "Registros utilizables": df_modelo.groupby("anio").size()
}).fillna(0).astype(int)

resumen_anual["Cobertura (%)"] = (
    resumen_anual["Registros utilizables"]
    / resumen_anual["Registros originales"] * 100
).round(2)

display(resumen_anual)

print(f"\nTotal de registros utilizables: {len(df_modelo):,}")

,Registros originales,Registros utilizables,Cobertura (%)
anio,,,
2019,2107,0,0.00
2020,2180,2046,93.85
2021,2176,2120,97.43
2022,2127,1967,92.48
2023,2111,2040,96.64
2024,2104,1987,94.44
2025,2130,1998,93.80



Total de registros utilizables: 12,158


In [4]:
# Definir ventanas expansivas para validación temporal
ventanas = [
    ([2020], 2021),
    ([2020, 2021], 2022),
    ([2020, 2021, 2022], 2023)
]

for i, (anios_train, anio_val) in enumerate(ventanas, start=1):
    train = df_modelo[df_modelo["anio"].isin(anios_train)]
    val = df_modelo[df_modelo["anio"] == anio_val]

    print(
        f"Fold {i}: "
        f"Entrenamiento {anios_train} ({len(train):,} registros) | "
        f"Validación {anio_val} ({len(val):,} registros)"
    )

Fold 1: Entrenamiento [2020] (2,046 registros) | Validación 2021 (2,120 registros)
Fold 2: Entrenamiento [2020, 2021] (4,166 registros) | Validación 2022 (1,967 registros)
Fold 3: Entrenamiento [2020, 2021, 2022] (6,133 registros) | Validación 2023 (2,040 registros)


## 4. Modelo de referencia: persistencia estacional

Se utiliza el rendimiento del mismo semestre del año anterior como
predicción directa. Este modelo no requiere entrenamiento y permite
establecer un punto de comparación para evaluar los modelos supervisados.

In [5]:
# Evaluar el baseline en las ventanas de validación temporal

resultados_baseline = []

for i, (anios_train, anio_val) in enumerate(ventanas, start=1):

    val = df_modelo[df_modelo["anio"] == anio_val]

    y_real = val["rendimiento"]
    y_pred = val["rendimiento_rezago"]

    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)

    resultados_baseline.append({
        "Fold": i,
        "Año validación": anio_val,
        "Registros": len(val),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

tabla_baseline = pd.DataFrame(resultados_baseline)

display(tabla_baseline.round(4))

print("\nPromedio de métricas entre folds:")
print(tabla_baseline[["MAE", "RMSE", "R2"]].mean().round(4))

,Fold,Año validación,Registros,MAE,RMSE,R2
0,1,2021,2120,1.1449,2.2728,-1.5115
1,2,2022,1967,0.4382,0.8629,0.6216
2,3,2023,2040,0.2546,0.6303,0.8032



Promedio de métricas entre folds:
MAE     0.6126
RMSE    1.2553
R2     -0.0289
dtype: float64


## 5. Definición de variables predictoras

Se utilizan variables territoriales, temporales, del sistema productivo
y el rendimiento histórico estacional.

Se excluyen la producción y el área cosechada del periodo objetivo,
porque no están disponibles antes de la siembra y generarían fuga
de información.

In [6]:
# Variables disponibles antes de la siembra
VARIABLES_NUMERICAS = [
    "anio",
    "semestre_b",
    "rendimiento_rezago"
]

VARIABLES_CATEGORICAS = [
    "departamento",
    "sistema_productivo"
]

VARIABLES_MODELO = VARIABLES_NUMERICAS + VARIABLES_CATEGORICAS
VARIABLE_OBJETIVO = "rendimiento"

# Seleccionar predictores y variable objetivo
X = df_modelo[VARIABLES_MODELO].copy()
y = df_modelo[VARIABLE_OBJETIVO].copy()

print("Variables predictoras:")
print(X.columns.tolist())

print(f"\nDimensiones de X: {X.shape}")
print(f"Dimensiones de y: {y.shape}")

print("\nTipos de datos:")
print(X.dtypes)

print("\nValores faltantes en predictores:")
print(X.isna().sum())

Variables predictoras:
['anio', 'semestre_b', 'rendimiento_rezago', 'departamento', 'sistema_productivo']

Dimensiones de X: (12158, 5)
Dimensiones de y: (12158,)

Tipos de datos:
anio                    int64
semestre_b              int64
rendimiento_rezago    float64
departamento              str
sistema_productivo        str
dtype: object

Valores faltantes en predictores:
anio                  0
semestre_b            0
rendimiento_rezago    0
departamento          0
sistema_productivo    0
dtype: int64


## 6. Pipeline de preprocesamiento

Se construye un pipeline para transformar las variables numéricas
y categóricas. Las transformaciones se ajustarán únicamente con
los datos de entrenamiento de cada partición temporal.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Transformaciones para las variables numéricas
transformador_numerico = StandardScaler()

# Transformaciones para las variables categóricas
transformador_categorico = OneHotEncoder(
    handle_unknown="ignore"
)

# Integración del preprocesamiento
preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", transformador_numerico, VARIABLES_NUMERICAS),
        ("categoricas", transformador_categorico, VARIABLES_CATEGORICAS)
    ]
)

print("Pipeline de preprocesamiento configurado correctamente.")
print(preprocesador)

Pipeline de preprocesamiento configurado correctamente.
ColumnTransformer(transformers=[('numericas', StandardScaler(),
                                 ['anio', 'semestre_b', 'rendimiento_rezago']),
                                ('categoricas',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['departamento', 'sistema_productivo'])])


## 7. Evaluación inicial de Ridge con validación temporal

Se evalúa un modelo Ridge utilizando tres ventanas expansivas.
En cada partición, el preprocesamiento y el modelo se ajustan
exclusivamente con los datos de entrenamiento.

Los resultados se compararán posteriormente con el modelo de
persistencia estacional.

In [8]:
from sklearn.linear_model import Ridge
from sklearn.base import clone

resultados_ridge = []

for i, (anios_train, anio_val) in enumerate(ventanas, start=1):

    # Selección temporal
    mask_train = df_modelo["anio"].isin(anios_train)
    mask_val = df_modelo["anio"] == anio_val

    X_train = X.loc[mask_train]
    y_train = y.loc[mask_train]

    X_val = X.loc[mask_val]
    y_val = y.loc[mask_val]

    # Pipeline independiente para cada fold
    modelo = Pipeline([
        ("preprocesamiento", clone(preprocesador)),
        ("regresor", Ridge(alpha=1.0))
    ])

    # Entrenamiento y predicción
    modelo.fit(X_train, y_train)
    predicciones = modelo.predict(X_val)

    # Evaluación
    resultados_ridge.append({
        "Fold": i,
        "Año validación": anio_val,
        "Registros": len(y_val),
        "MAE": mean_absolute_error(y_val, predicciones),
        "RMSE": np.sqrt(mean_squared_error(y_val, predicciones)),
        "R2": r2_score(y_val, predicciones)
    })

tabla_ridge = pd.DataFrame(resultados_ridge)

display(tabla_ridge.round(4))

print("\nPromedio de métricas entre folds:")
print(tabla_ridge[["MAE", "RMSE", "R2"]].mean().round(4))

,Fold,Año validación,Registros,MAE,RMSE,R2
0,1,2021,2120,1.0763,1.8111,-0.5948
1,2,2022,1967,1.0212,1.2269,0.2351
2,3,2023,2040,0.5304,0.7686,0.7075



Promedio de métricas entre folds:
MAE     0.8760
RMSE    1.2689
R2      0.1159
dtype: float64


## 8. Optimización de hiperparámetros de Ridge

Se evalúan diferentes valores de regularización `alpha` mediante
validación temporal con ventanas expansivas. La selección se realiza
utilizando el RMSE promedio de las tres particiones, sin consultar
los conjuntos reservados de 2024 y 2025.

In [9]:
# Valores de regularización que vamos a evaluar
valores_alpha = [100, 300, 1000, 3000, 10000]

resultados_alpha = []

for alpha in valores_alpha:

    errores_rmse = []
    errores_mae = []

    for anios_train, anio_val in ventanas:

        mask_train = df_modelo["anio"].isin(anios_train)
        mask_val = df_modelo["anio"] == anio_val

        modelo = Pipeline([
            ("preprocesamiento", clone(preprocesador)),
            ("regresor", Ridge(alpha=alpha))
        ])

        modelo.fit(X.loc[mask_train], y.loc[mask_train])

        pred = modelo.predict(X.loc[mask_val])

        errores_rmse.append(
            np.sqrt(mean_squared_error(y.loc[mask_val], pred))
        )
        errores_mae.append(
            mean_absolute_error(y.loc[mask_val], pred)
        )

    resultados_alpha.append({
        "alpha": alpha,
        "RMSE_promedio": np.mean(errores_rmse),
        "MAE_promedio": np.mean(errores_mae),
        "RMSE_2021": errores_rmse[0],
        "RMSE_2022": errores_rmse[1],
        "RMSE_2023": errores_rmse[2]
    })

tabla_alpha = pd.DataFrame(resultados_alpha)
tabla_alpha = tabla_alpha.sort_values("RMSE_promedio")

display(tabla_alpha.round(4))

mejor_alpha = tabla_alpha.iloc[0]["alpha"]

print(f"\nMejor alpha: {mejor_alpha}")
print(f"RMSE promedio mínimo: {tabla_alpha.iloc[0]['RMSE_promedio']:.4f}")

,alpha,RMSE_promedio,MAE_promedio,RMSE_2021,RMSE_2022,RMSE_2023
3,3000,1.1296,0.8331,1.4067,1.0802,0.9019
2,1000,1.1447,0.8107,1.5015,1.1245,0.8082
1,300,1.2065,0.8361,1.6586,1.1859,0.7750
0,100,1.2401,0.8525,1.7423,1.2106,0.7674
4,10000,1.2442,0.9827,1.4591,1.1766,1.0968



Mejor alpha: 3000.0
RMSE promedio mínimo: 1.1296


## 9. Evaluación inicial de Random Forest

Se evalúa Random Forest mediante validación temporal con ventanas
expansivas, utilizando las mismas variables y particiones empleadas
para Ridge.

Esta configuración inicial permitirá cuantificar posteriormente
el beneficio de la optimización de hiperparámetros.

In [10]:
from sklearn.ensemble import RandomForestRegressor

resultados_rf = []

for i, (anios_train, anio_val) in enumerate(ventanas, start=1):

    mask_train = df_modelo["anio"].isin(anios_train)
    mask_val = df_modelo["anio"] == anio_val

    modelo_rf = Pipeline([
        ("preprocesamiento", clone(preprocesador)),
        ("regresor", RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ])

    modelo_rf.fit(X.loc[mask_train], y.loc[mask_train])
    pred = modelo_rf.predict(X.loc[mask_val])

    resultados_rf.append({
        "Fold": i,
        "Año validación": anio_val,
        "Registros": mask_val.sum(),
        "MAE": mean_absolute_error(y.loc[mask_val], pred),
        "RMSE": np.sqrt(mean_squared_error(y.loc[mask_val], pred)),
        "R2": r2_score(y.loc[mask_val], pred)
    })

tabla_rf = pd.DataFrame(resultados_rf)

display(tabla_rf.round(4))

print("\nPromedio de métricas entre folds:")
print(tabla_rf[["MAE", "RMSE", "R2"]].mean().round(4))

,Fold,Año validación,Registros,MAE,RMSE,R2
0,1,2021,2120,1.1310,2.0031,-0.9508
1,2,2022,1967,0.6164,1.0146,0.4769
2,3,2023,2040,0.3987,0.6875,0.7659



Promedio de métricas entre folds:
MAE     0.7154
RMSE    1.2351
R2      0.0973
dtype: float64


## 10. Optimización de hiperparámetros de Random Forest

Se utiliza una búsqueda de configuraciones mediante validación
temporal con ventanas expansivas.

La selección se realiza minimizando el RMSE promedio de los
años 2021, 2022 y 2023, sin utilizar los datos reservados
de 2024 y 2025.

In [11]:
from sklearn.model_selection import ParameterGrid

# Espacio inicial de búsqueda
parametros_rf = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 3, 5]
}

resultados_busqueda_rf = []

for params in ParameterGrid(parametros_rf):

    errores_rmse = []
    errores_mae = []

    for anios_train, anio_val in ventanas:

        mask_train = df_modelo["anio"].isin(anios_train)
        mask_val = df_modelo["anio"] == anio_val

        modelo = Pipeline([
            ("preprocesamiento", clone(preprocesador)),
            ("regresor", RandomForestRegressor(
                **params,
                random_state=42,
                n_jobs=-1
            ))
        ])

        modelo.fit(X.loc[mask_train], y.loc[mask_train])
        pred = modelo.predict(X.loc[mask_val])

        errores_rmse.append(
            np.sqrt(mean_squared_error(y.loc[mask_val], pred))
        )
        errores_mae.append(
            mean_absolute_error(y.loc[mask_val], pred)
        )

    resultados_busqueda_rf.append({
        **params,
        "RMSE_promedio": np.mean(errores_rmse),
        "MAE_promedio": np.mean(errores_mae),
        "RMSE_2021": errores_rmse[0],
        "RMSE_2022": errores_rmse[1],
        "RMSE_2023": errores_rmse[2]
    })

tabla_busqueda_rf = (
    pd.DataFrame(resultados_busqueda_rf)
    .sort_values("RMSE_promedio")
    .reset_index(drop=True)
)

display(tabla_busqueda_rf.head(10).round(4))

mejores_parametros_rf = {
    "n_estimators": int(tabla_busqueda_rf.iloc[0]["n_estimators"]),
    "max_depth": (
        None if pd.isna(tabla_busqueda_rf.iloc[0]["max_depth"])
        else int(tabla_busqueda_rf.iloc[0]["max_depth"])
    ),
    "min_samples_leaf": int(tabla_busqueda_rf.iloc[0]["min_samples_leaf"])
}

print("\nMejores hiperparámetros:")
print(mejores_parametros_rf)

print(
    f"RMSE promedio mínimo: "
    f"{tabla_busqueda_rf.iloc[0]['RMSE_promedio']:.4f}"
)

,max_depth,min_samples_leaf,n_estimators,RMSE_promedio,MAE_promedio,RMSE_2021,RMSE_2022,RMSE_2023
0,5.0,5,200,1.1005,0.6798,1.8094,0.8375,0.6545
1,10.0,5,200,1.1011,0.6649,1.8299,0.8377,0.6356
2,5.0,3,200,1.1071,0.6817,1.8254,0.8396,0.6562
3,NaN,5,200,1.1084,0.6764,1.8328,0.8464,0.6458
4,10.0,5,100,1.1088,0.6663,1.8494,0.8413,0.6357
5,5.0,5,100,1.1089,0.6821,1.8323,0.8411,0.6535
6,10.0,3,200,1.1110,0.6651,1.8629,0.8372,0.6330
7,5.0,3,100,1.1143,0.6843,1.8432,0.8437,0.6558
8,NaN,5,100,1.1163,0.6783,1.8522,0.8497,0.6468
9,10.0,3,100,1.1177,0.6664,1.8761,0.8424,0.6346



Mejores hiperparámetros:
{'n_estimators': 200, 'max_depth': 5, 'min_samples_leaf': 5}
RMSE promedio mínimo: 1.1005


## 11. Evaluación inicial de XGBoost

Se evalúa XGBoost mediante validación temporal con ventanas expansivas,
utilizando las mismas variables predictoras y particiones de los
modelos anteriores.

Esta evaluación permitirá establecer el desempeño inicial del modelo
antes de optimizar sus hiperparámetros.

In [12]:
from xgboost import XGBRegressor

resultados_xgb = []

for i, (anios_train, anio_val) in enumerate(ventanas, start=1):

    mask_train = df_modelo["anio"].isin(anios_train)
    mask_val = df_modelo["anio"] == anio_val

    modelo_xgb = Pipeline([
        ("preprocesamiento", clone(preprocesador)),
        ("regresor", XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ])

    modelo_xgb.fit(X.loc[mask_train], y.loc[mask_train])
    pred = modelo_xgb.predict(X.loc[mask_val])

    resultados_xgb.append({
        "Fold": i,
        "Año validación": anio_val,
        "Registros": mask_val.sum(),
        "MAE": mean_absolute_error(y.loc[mask_val], pred),
        "RMSE": np.sqrt(mean_squared_error(y.loc[mask_val], pred)),
        "R2": r2_score(y.loc[mask_val], pred)
    })

tabla_xgb = pd.DataFrame(resultados_xgb)

display(tabla_xgb.round(4))

print("\nPromedio de métricas entre folds:")
print(tabla_xgb[["MAE", "RMSE", "R2"]].mean().round(4))

,Fold,Año validación,Registros,MAE,RMSE,R2
0,1,2021,2120,1.0919,2.0075,-0.9595
1,2,2022,1967,0.5527,0.9280,0.5624
2,3,2023,2040,0.3861,0.6346,0.8006



Promedio de métricas entre folds:
MAE     0.6769
RMSE    1.1900
R2      0.1345
dtype: float64


## 12. Optimización de hiperparámetros de XGBoost

Se realiza una búsqueda de hiperparámetros mediante validación temporal
con ventanas expansivas.

La selección se basa en el RMSE promedio de los años 2021–2023,
manteniendo los años 2024 y 2025 fuera del proceso de optimización.

In [13]:
# Espacio de búsqueda de XGBoost
parametros_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0]
}

resultados_busqueda_xgb = []

for params in ParameterGrid(parametros_xgb):

    errores_rmse = []
    errores_mae = []

    for anios_train, anio_val in ventanas:

        mask_train = df_modelo["anio"].isin(anios_train)
        mask_val = df_modelo["anio"] == anio_val

        modelo = Pipeline([
            ("preprocesamiento", clone(preprocesador)),
            ("regresor", XGBRegressor(
                **params,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            ))
        ])

        modelo.fit(X.loc[mask_train], y.loc[mask_train])
        pred = modelo.predict(X.loc[mask_val])

        errores_rmse.append(
            np.sqrt(mean_squared_error(y.loc[mask_val], pred))
        )
        errores_mae.append(
            mean_absolute_error(y.loc[mask_val], pred)
        )

    resultados_busqueda_xgb.append({
        **params,
        "RMSE_promedio": np.mean(errores_rmse),
        "MAE_promedio": np.mean(errores_mae),
        "RMSE_2021": errores_rmse[0],
        "RMSE_2022": errores_rmse[1],
        "RMSE_2023": errores_rmse[2]
    })

tabla_busqueda_xgb = (
    pd.DataFrame(resultados_busqueda_xgb)
    .sort_values("RMSE_promedio")
    .reset_index(drop=True)
)

display(tabla_busqueda_xgb.head(10).round(4))

mejores_parametros_xgb = {
    "n_estimators": int(tabla_busqueda_xgb.iloc[0]["n_estimators"]),
    "max_depth": int(tabla_busqueda_xgb.iloc[0]["max_depth"]),
    "learning_rate": float(tabla_busqueda_xgb.iloc[0]["learning_rate"]),
    "subsample": float(tabla_busqueda_xgb.iloc[0]["subsample"])
}

print("\nMejores hiperparámetros:")
print(mejores_parametros_xgb)

print(
    f"RMSE promedio mínimo: "
    f"{tabla_busqueda_xgb.iloc[0]['RMSE_promedio']:.4f}"
)

,learning_rate,max_depth,n_estimators,subsample,RMSE_promedio,MAE_promedio,RMSE_2021,RMSE_2022,RMSE_2023
0,0.10,3,100,1.0,1.1056,0.6674,1.8848,0.7927,0.6393
1,0.05,3,200,1.0,1.1086,0.6683,1.8879,0.7967,0.6412
2,0.05,3,100,1.0,1.1089,0.6830,1.8479,0.8127,0.6661
3,0.05,3,100,0.8,1.1104,0.6808,1.8653,0.8082,0.6576
4,0.10,3,200,1.0,1.1171,0.6706,1.9120,0.7978,0.6415
5,0.10,3,100,0.8,1.1210,0.6718,1.9221,0.7970,0.6440
6,0.05,3,200,0.8,1.1243,0.6725,1.9345,0.7975,0.6410
7,0.10,3,200,0.8,1.1362,0.6742,1.9672,0.7989,0.6425
8,0.05,5,100,0.8,1.1375,0.6718,1.9237,0.8618,0.6271
9,0.05,5,100,1.0,1.1408,0.6730,1.9326,0.8609,0.6290



Mejores hiperparámetros:
{'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 1.0}
RMSE promedio mínimo: 1.1056


## 13. Evaluación de los modelos seleccionados en 2024

Se entrenan los modelos con los datos de 2020–2023 y se evalúan
sobre 2024, utilizando los hiperparámetros seleccionados mediante
validación temporal.

El año 2025 permanece reservado para la evaluación final independiente.

In [14]:
# Separación temporal
mask_train = df_modelo["anio"].between(2020, 2023)
mask_val = df_modelo["anio"] == 2024

X_train = X.loc[mask_train]
y_train = y.loc[mask_train]

X_val = X.loc[mask_val]
y_val = y.loc[mask_val]

# Modelos seleccionados
modelos_finalistas = {
    "Ridge optimizado": Ridge(alpha=3000),

    "Random Forest optimizado": RandomForestRegressor(
        **mejores_parametros_rf,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost optimizado": XGBRegressor(
        **mejores_parametros_xgb,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

resultados_2024 = []

# Baseline de persistencia
pred_baseline = df_modelo.loc[
    mask_val, "rendimiento_rezago"
]

resultados_2024.append({
    "Modelo": "Persistencia estacional",
    "MAE": mean_absolute_error(y_val, pred_baseline),
    "RMSE": np.sqrt(mean_squared_error(y_val, pred_baseline)),
    "R2": r2_score(y_val, pred_baseline)
})

# Evaluación de modelos supervisados
for nombre, estimador in modelos_finalistas.items():

    modelo = Pipeline([
        ("preprocesamiento", clone(preprocesador)),
        ("regresor", estimador)
    ])

    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_val)

    resultados_2024.append({
        "Modelo": nombre,
        "MAE": mean_absolute_error(y_val, pred),
        "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
        "R2": r2_score(y_val, pred)
    })

tabla_2024 = (
    pd.DataFrame(resultados_2024)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print(f"Registros de entrenamiento: {len(y_train):,}")
print(f"Registros de validación 2024: {len(y_val):,}")

display(tabla_2024.round(4))

Registros de entrenamiento: 8,173
Registros de validación 2024: 1,987


,Modelo,MAE,RMSE,R2
0,XGBoost optimizado,0.3536,0.6184,0.8064
1,Random Forest optimizado,0.3667,0.6340,0.7965
2,Persistencia estacional,0.2470,0.6591,0.7800
3,Ridge optimizado,0.5710,0.8064,0.6707


## 14. Verificación de consistencia temporal del rezago

Se verifica que el rendimiento histórico utilizado como predictor
corresponda al mismo semestre del año anterior, sin incorporar
información del periodo objetivo.

In [15]:
# Verificar que cada rezago corresponda al mismo semestre del año anterior
claves = ["cod_municipio", "sistema_productivo", "semestre_b"]

historico = df_modelo[
    claves + ["anio", "rendimiento"]
].copy()

historico["anio"] += 1
historico = historico.rename(
    columns={"rendimiento": "rezago_verificacion"}
)

verificacion = df_modelo.merge(
    historico,
    on=claves + ["anio"],
    how="left",
    validate="one_to_one"
)

coinciden = np.isclose(
    verificacion["rendimiento_rezago"],
    verificacion["rezago_verificacion"],
    equal_nan=False
)

print(f"Registros verificados: {len(verificacion):,}")
print(f"Rezagos correctos: {coinciden.sum():,}")
print(f"Rezagos inconsistentes: {(~coinciden).sum():,}")

Registros verificados: 12,158
Rezagos correctos: 9,628
Rezagos inconsistentes: 2,530


In [16]:
# Verificar los rezagos utilizando el histórico completo
claves = ["cod_municipio", "sistema_productivo", "semestre_b"]

# Recuperar todos los rendimientos históricos, incluido 2019
historico_completo = df[
    claves + ["anio", "rendimiento"]
].copy()

# Desplazar cada rendimiento un año hacia adelante
historico_completo["anio"] += 1

historico_completo = historico_completo.rename(
    columns={"rendimiento": "rezago_verificacion"}
)

# Comparar contra los registros utilizados para modelar
verificacion = df_modelo.merge(
    historico_completo,
    on=claves + ["anio"],
    how="left",
    validate="one_to_one"
)

coinciden = np.isclose(
    verificacion["rendimiento_rezago"],
    verificacion["rezago_verificacion"],
    equal_nan=False
)

print(f"Registros verificados: {len(verificacion):,}")
print(f"Rezagos correctos: {coinciden.sum():,}")
print(f"Rezagos inconsistentes: {(~coinciden).sum():,}")

print("\nInconsistencias por año:")
print(
    verificacion.loc[~coinciden]
    .groupby("anio")
    .size()
)

Registros verificados: 12,158
Rezagos correctos: 12,158
Rezagos inconsistentes: 0

Inconsistencias por año:
Series([], dtype: int64)


## 15. Evaluación final del modelo seleccionado

Se entrena XGBoost con los datos de 2020–2024, utilizando los
hiperparámetros seleccionados previamente.

El año 2025 se utiliza exclusivamente como conjunto de prueba
independiente. Se compara el desempeño contra la persistencia
estacional mediante MAE, RMSE y R².

In [17]:
# Separación temporal definitiva
mask_train_final = df_modelo["anio"].between(2020, 2024)
mask_test_final = df_modelo["anio"] == 2025

X_train_final = X.loc[mask_train_final]
y_train_final = y.loc[mask_train_final]

X_test_final = X.loc[mask_test_final]
y_test_final = y.loc[mask_test_final]

# XGBoost seleccionado en la validación de 2024
modelo_final = Pipeline([
    ("preprocesamiento", clone(preprocesador)),
    ("regresor", XGBRegressor(
        **mejores_parametros_xgb,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

modelo_final.fit(X_train_final, y_train_final)

pred_xgb_final = modelo_final.predict(X_test_final)

# Persistencia estacional
pred_baseline_final = df_modelo.loc[
    mask_test_final, "rendimiento_rezago"
]

# Comparación final
resultados_finales = []

for nombre, pred in [
    ("Persistencia estacional", pred_baseline_final),
    ("XGBoost optimizado", pred_xgb_final)
]:
    resultados_finales.append({
        "Modelo": nombre,
        "MAE": mean_absolute_error(y_test_final, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test_final, pred)),
        "R2": r2_score(y_test_final, pred)
    })

tabla_final = pd.DataFrame(resultados_finales)

print(f"Registros de entrenamiento: {len(y_train_final):,}")
print(f"Registros de prueba 2025: {len(y_test_final):,}")

display(tabla_final.round(4))

Registros de entrenamiento: 10,160
Registros de prueba 2025: 1,998


,Modelo,MAE,RMSE,R2
0,Persistencia estacional,0.1655,0.4494,0.8936
1,XGBoost optimizado,0.2719,0.4415,0.8973


## 16. Registro y seguimiento de experimentos con MLflow

Se utiliza MLflow para registrar los hiperparámetros, métricas
y artefactos de los modelos evaluados, garantizando la trazabilidad
y reproducibilidad de los experimentos.

In [35]:
import os
import mlflow

# Configuración del servidor MLflow
tracking_uri = os.getenv(
    "MLFLOW_TRACKING_URI",
    "http://localhost:5000"
)

nombre_experimento = os.getenv(
    "MLFLOW_EXPERIMENT",
    "agropredict-maiz-rendimiento"
)

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(nombre_experimento)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", nombre_experimento)

Tracking URI: http://localhost:5000
Experimento: agropredict-maiz-rendimiento


In [25]:
with mlflow.start_run(run_name="XGBoost_optimizado"):

    # Identificación del experimento
    mlflow.set_tags({
        "modelo": "XGBoost",
        "proyecto": "AgroPredict Colombia",
        "validacion": "Temporal expansiva",
        "variable_objetivo": "rendimiento_t_ha"
    })

    # Hiperparámetros seleccionados
    mlflow.log_params(mejores_parametros_xgb)

    # Métricas de validación cruzada (2021-2023)
    mlflow.log_metric("cv_rmse", 1.1056)
    mlflow.log_metric("cv_mae", 0.6674)

    # Métricas de selección (2024)
    mlflow.log_metric("val_2024_rmse", 0.6184)
    mlflow.log_metric("val_2024_mae", 0.3536)
    mlflow.log_metric("val_2024_r2", 0.8064)

    # Métricas finales (2025)
    mlflow.log_metric("test_2025_rmse", 0.4415)
    mlflow.log_metric("test_2025_mae", 0.2719)
    mlflow.log_metric("test_2025_r2", 0.8973)

print("Experimento XGBoost registrado correctamente en MLflow.")

🏃 View run XGBoost_optimizado at: http://localhost:5000/#/experiments/1/runs/db2ce31c70f1409da62e27900ced1ddd
🧪 View experiment at: http://localhost:5000/#/experiments/1
Experimento XGBoost registrado correctamente en MLflow.


In [26]:
with mlflow.start_run(run_name="Baseline_persistencia_estacional"):

    mlflow.set_tags({
        "modelo": "Persistencia estacional",
        "proyecto": "AgroPredict Colombia",
        "tipo": "Benchmark",
        "validacion": "Temporal expansiva"
    })

    # Configuración del benchmark
    mlflow.log_param(
        "estrategia",
        "Rendimiento del mismo semestre del año anterior"
    )

    # Validación cruzada (2021-2023)
    mlflow.log_metric("cv_rmse", 1.2553)
    mlflow.log_metric("cv_mae", 0.6126)

    # Validación de selección (2024)
    mlflow.log_metric("val_2024_rmse", 0.6591)
    mlflow.log_metric("val_2024_mae", 0.2470)
    mlflow.log_metric("val_2024_r2", 0.7800)

    # Evaluación final (2025)
    mlflow.log_metric("test_2025_rmse", 0.4494)
    mlflow.log_metric("test_2025_mae", 0.1655)
    mlflow.log_metric("test_2025_r2", 0.8936)

print("Benchmark registrado correctamente en MLflow.")

🏃 View run Baseline_persistencia_estacional at: http://localhost:5000/#/experiments/1/runs/6b86b4e83726408094fcd3a0060ffaed
🧪 View experiment at: http://localhost:5000/#/experiments/1
Benchmark registrado correctamente en MLflow.


In [27]:
with mlflow.start_run(run_name="RandomForest_optimizado"):

    mlflow.set_tags({
        "modelo": "Random Forest",
        "proyecto": "AgroPredict Colombia",
        "validacion": "Temporal expansiva",
        "tipo": "Modelo candidato"
    })

    # Hiperparámetros óptimos
    mlflow.log_params(mejores_parametros_rf)

    # Validación cruzada (2021-2023)
    mlflow.log_metric("cv_rmse", 1.1005)
    mlflow.log_metric("cv_mae", 0.6798)

    # Validación de selección (2024)
    mlflow.log_metric("val_2024_rmse", 0.6340)
    mlflow.log_metric("val_2024_mae", 0.3667)
    mlflow.log_metric("val_2024_r2", 0.7965)

print("Random Forest optimizado registrado correctamente en MLflow.")

🏃 View run RandomForest_optimizado at: http://localhost:5000/#/experiments/1/runs/1c92678281af4819848f4b55a0c92bed
🧪 View experiment at: http://localhost:5000/#/experiments/1
Random Forest optimizado registrado correctamente en MLflow.


In [28]:
with mlflow.start_run(run_name="Ridge_optimizado"):

    mlflow.set_tags({
        "modelo": "Ridge",
        "proyecto": "AgroPredict Colombia",
        "validacion": "Temporal expansiva",
        "tipo": "Modelo candidato"
    })

    # Hiperparámetro óptimo
    mlflow.log_param("alpha", 3000)

    # Validación cruzada (2021-2023)
    mlflow.log_metric("cv_rmse", 1.1296)
    mlflow.log_metric("cv_mae", 0.8331)

    # Validación de selección (2024)
    mlflow.log_metric("val_2024_rmse", 0.8064)
    mlflow.log_metric("val_2024_mae", 0.5710)
    mlflow.log_metric("val_2024_r2", 0.6707)

print("Ridge optimizado registrado correctamente en MLflow.")

🏃 View run Ridge_optimizado at: http://localhost:5000/#/experiments/1/runs/063a4012200d4c1a9a3777cb48ac3156
🧪 View experiment at: http://localhost:5000/#/experiments/1
Ridge optimizado registrado correctamente en MLflow.


In [32]:
import mlflow
import mlflow.sklearn

# Mantener la conexión con nuestro servidor MLflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("agropredict-maiz-rendimiento")

with mlflow.start_run(run_name="XGBoost_modelo_final_2025"):

    # Información del modelo
    mlflow.set_tags({
        "modelo": "XGBoost",
        "proyecto": "AgroPredict Colombia",
        "tipo": "Modelo final",
        "entrenamiento": "2020-2024",
        "prueba": "2025"
    })

    # Registrar hiperparámetros
    mlflow.log_params(mejores_parametros_xgb)

    # Registrar métricas finales
    mlflow.log_metric("test_2025_rmse", 0.4415)
    mlflow.log_metric("test_2025_mae", 0.2719)
    mlflow.log_metric("test_2025_r2", 0.8973)

    # Guardar el pipeline completo
    info_modelo = mlflow.sklearn.log_model(
    sk_model=modelo_final,
    name="modelo_xgboost_maiz",
    skops_trusted_types=[
        "xgboost.core.Booster",
        "xgboost.sklearn.XGBRegressor"
    ]
    )

print("Modelo final registrado correctamente.")
print("URI del modelo:", info_modelo.model_uri)

🏃 View run XGBoost_modelo_final_2025 at: http://localhost:5000/#/experiments/1/runs/207c984beb084efd8dca32677b249ea3
🧪 View experiment at: http://localhost:5000/#/experiments/1
Modelo final registrado correctamente.
URI del modelo: models:/m-82d0c28827314c0fb9cd5a6ecb6944bb


### 16.1. Verificación de integridad del modelo serializado

Se recupera el pipeline final de XGBoost desde MLflow y se
comparan sus predicciones con las del modelo original sobre
10 observaciones.

La diferencia máxima absoluta obtenida fue de 0.0,
confirmando que el proceso de serialización y recuperación
conserva el comportamiento predictivo del modelo.

In [36]:
import numpy as np
import mlflow
import skops.io as sio
from pathlib import Path

# URI generada al registrar el modelo en MLflow
uri_modelo = info_modelo.model_uri

# Descargar los archivos del modelo desde MLflow
ruta_modelo = mlflow.artifacts.download_artifacts(
    artifact_uri=uri_modelo
)

# Localizar el archivo serializado
archivo_modelo = Path(ruta_modelo) / "model.skops"

print("Archivo encontrado:", archivo_modelo.exists())

# Cargar el pipeline autorizando los tipos conocidos
modelo_recuperado = sio.load(
    archivo_modelo,
    trusted=[
        "xgboost.core.Booster",
        "xgboost.sklearn.XGBRegressor"
    ]
)

# Comparar las predicciones de ambos modelos
X_verificacion = X.head(10)

predicciones_originales = modelo_final.predict(X_verificacion)
predicciones_recuperadas = modelo_recuperado.predict(X_verificacion)

print("\n¿Las predicciones coinciden?")
print(np.allclose(predicciones_originales, predicciones_recuperadas))

print("\nDiferencia máxima absoluta:")
print(np.max(np.abs(
    predicciones_originales - predicciones_recuperadas
)))

Archivo encontrado: True

¿Las predicciones coinciden?
True

Diferencia máxima absoluta:
0.0
